# Classificador de 1 a 5, V/F, A-E — Corretor de Multiprova

Este notebook implementa o pipeline pedido:

1. Filtragem da base **EMNIST** para gerar 3 datasets de treino/teste:
   - Binário **Verdadeiro/Falso** (letras `V` e `F`)
   - Multiclasse **dígitos 1 a 5**
   - Multiclasse **letras A a E**
2. Procedimento inicial nos moldes do classificador binário do dígito 5 feito em sala (achatar 28x28 -> vetor, normalizar, treinar/testar).
3. Treino de 5 classificadores: `RandomForestClassifier`, `GaussianNB`, `LogisticRegression`, `SVC`, `MLPClassifier`.
4. Predição no conjunto de teste para cada modelo.
5. `classification_report` + exportação de um `.csv` por modelo, e um resumo consolidado.
6. Ranking dos modelos, priorizando acurácia alta **e** pouca diferença entre treino e teste (baixo overfitting).
7. Exportação dos melhores modelos (`joblib`) para uso no app Streamlit com os 3 canvas (`app_streamlit/app.py`).

> **Contexto:** os 3 datasets simulam a leitura de respostas de uma folha de gabarito (multiprova): questões V/F, questões com alternativas numéricas 1-5 e questões com alternativas A-E.


## 0. Instalação de dependências
Rode esta célula apenas uma vez por sessão do Colab.

In [ ]:
!pip install scikit-learn pandas numpy matplotlib joblib -q

In [ ]:
import os
import gzip
import string
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)

## 1. Carregar a base EMNIST

> **Nota sobre o erro no Colab:** o pacote `emnist` do PyPI tenta baixar o dataset
> primeiro de um link do Google Drive (que hoje devolve uma página HTML de aviso
> em vez do arquivo, causando `BadZipFile: File is not a zip file`) e, se isso falhar,
> de uma URL antiga da NIST (`itl.nist.gov`) que está fora do ar. Por isso a célula
> abaixo **não usa o pacote `emnist`**: ela baixa o zip oficial diretamente do espelho
> atual da NIST (`biometrics.nist.gov`), enviando um `User-Agent` de navegador (o
> servidor bloqueia requisições sem esse cabeçalho, que é o padrão do Colab/`urllib`)
> e faz o parsing dos arquivos IDX manualmente (mesma lógica usada internamente pelo
> pacote `emnist`).

- `digits`: dígitos 0-9 (rótulos já vêm como 0-9)
- `letters`: letras a-z com maiúscula/minúscula mescladas (rótulos 1-26 ou 0-25, dependendo do arquivo — por isso vamos **verificar** abaixo em vez de supor).

In [ ]:
EMNIST_ZIP_URL = 'https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip'
EMNIST_ZIP_PATH = os.path.expanduser('~/.cache/emnist_manual/emnist_gzip.zip')


def _baixar_zip_emnist(destino=EMNIST_ZIP_PATH, url=EMNIST_ZIP_URL, tentativas=3):
    """Baixa o zip oficial do EMNIST (NIST), usando um User-Agent de navegador para
    evitar o bloqueio de requisições "de robô" que causa o erro visto com o pacote
    `emnist`. Faz cache local (~560MB) para não baixar de novo em execuções futuras."""
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    if os.path.isfile(destino) and os.path.getsize(destino) > 5 * 10**8:
        print('Zip do EMNIST já em cache:', destino)
        return destino

    parcial = destino + '.partial'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    ultimo_erro = None
    for tentativa in range(1, tentativas + 1):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=60) as resp, open(parcial, 'wb') as f:
                total = int(resp.headers.get('Content-Length', 0))
                lido = 0
                bloco = 1024 * 1024
                while True:
                    chunk = resp.read(bloco)
                    if not chunk:
                        break
                    f.write(chunk)
                    lido += len(chunk)
                    if total:
                        print(f'\rBaixando EMNIST: {lido / 1e6:.0f}MB / {total / 1e6:.0f}MB', end='')
            print()
            os.replace(parcial, destino)
            return destino
        except Exception as e:
            ultimo_erro = e
            if os.path.isfile(parcial):
                os.remove(parcial)
            print(f'Tentativa {tentativa} falhou: {e}')
    raise RuntimeError(f'Não foi possível baixar o EMNIST após {tentativas} tentativas') from ultimo_erro


def _ler_idx_gz(zip_ref, caminho_interno):
    """Lê um arquivo idx-ubyte.gz de dentro do zip do EMNIST e retorna um array numpy
    (mesmo formato usado pelo MNIST: http://yann.lecun.com/exdb/mnist/)."""
    with zip_ref.open(caminho_interno) as f_zip:
        dados = gzip.decompress(f_zip.read())
    tipos = {0x08: np.uint8, 0x09: np.int8, 0x0B: np.int16, 0x0C: np.int32, 0x0D: np.float32, 0x0E: np.float64}
    n_dims = dados[3]
    dtype = tipos[dados[2]]
    shape = [int.from_bytes(dados[4 + 4 * i:8 + 4 * i], 'big') for i in range(n_dims)]
    offset = 4 + 4 * n_dims
    return np.frombuffer(dados, dtype=dtype, offset=offset).reshape(shape)


def carregar_emnist(dataset, usage):
    """Substitui `extract_training_samples`/`extract_test_samples` do pacote `emnist`.
    `dataset`: 'letters' ou 'digits'. `usage`: 'train' ou 'test'.
    Retorna (imagens, rotulos), com imagens já na orientação correta ("em pé")."""
    zip_path = _baixar_zip_emnist()
    with zipfile.ZipFile(zip_path) as zf:
        imagens = _ler_idx_gz(zf, f'gzip/emnist-{dataset}-{usage}-images-idx3-ubyte.gz')
        rotulos = _ler_idx_gz(zf, f'gzip/emnist-{dataset}-{usage}-labels-idx1-ubyte.gz')
    # O arquivo original da NIST vem com largura/altura trocadas em relação ao MNIST;
    # corrigimos aqui (mesma correção feita internamente pelo pacote `emnist`).
    imagens = imagens.swapaxes(1, 2)
    return imagens, rotulos

In [ ]:
letters_train_x, letters_train_y = carregar_emnist('letters', 'train')
letters_test_x, letters_test_y = carregar_emnist('letters', 'test')

digits_train_x, digits_train_y = carregar_emnist('digits', 'train')
digits_test_x, digits_test_y = carregar_emnist('digits', 'test')

print('letters treino:', letters_train_x.shape, letters_train_y.shape)
print('letters teste :', letters_test_x.shape, letters_test_y.shape)
print('digits  treino:', digits_train_x.shape, digits_train_y.shape)
print('digits  teste :', digits_test_x.shape, digits_test_y.shape)

## 2. Descobrir o mapeamento rótulo → letra e checar orientação

O pacote `emnist` pode indexar as letras a partir de 0 ou de 1 dependendo da versão.
Em vez de fixar isso "no chute", detectamos o offset automaticamente e conferimos
visualmente se a imagem está na orientação correta (dígito/letra "em pé", como no MNIST).

In [ ]:
rotulos_letras_unicos = sorted(np.unique(letters_train_y))
offset = rotulos_letras_unicos[0]  # 0 se a-z começa em 0, 1 se começa em 1
mapa_letras = {r: string.ascii_uppercase[r - offset] for r in rotulos_letras_unicos}
print('offset detectado:', offset)
print('mapa rótulo -> letra:', mapa_letras)


In [ ]:
# Conferência visual: plota algumas amostras de letras e dígitos com seus rótulos.
# Se as imagens aparecerem giradas/espelhadas, descomente a linha de transpose indicada.

fig, eixos = plt.subplots(2, 6, figsize=(12, 4))

for i, ax in enumerate(eixos[0]):
    img = letters_train_x[i]
    # img = img.T  # <- descomente se a letra aparecer deitada/espelhada
    ax.imshow(img, cmap='gray')
    ax.set_title(mapa_letras[letters_train_y[i]])
    ax.axis('off')

for i, ax in enumerate(eixos[1]):
    img = digits_train_x[i]
    # img = img.T  # <- descomente se o dígito aparecer deitado/espelhado
    ax.imshow(img, cmap='gray')
    ax.set_title(str(digits_train_y[i]))
    ax.axis('off')

plt.tight_layout()
plt.show()

print('>>> Se as letras/dígitos acima estiverem legíveis e "em pé", está tudo certo.')
print('>>> Caso contrário, ative a linha "img = img.T" nas duas células de construção de dataset abaixo.')


## 3. Montar os 3 datasets

Reaproveitamos a mesma lógica do classificador binário do dígito 5 feito em sala:
1. Filtrar as classes desejadas.
2. Achatar cada imagem 28x28 em um vetor de 784 posições.
3. Normalizar os pixels para o intervalo [0, 1].
4. (Opcional) Sub-amostrar de forma estratificada para o treino rodar em tempo razoável no Colab.

In [ ]:
TRANSPOR_IMAGENS = False  # mude para True se a célula de conferência acima mostrou imagens erradas

def achatar_e_normalizar(X_imgs, transpor=TRANSPOR_IMAGENS):
    if transpor:
        X_imgs = np.transpose(X_imgs, (0, 2, 1))
    return X_imgs.reshape(len(X_imgs), -1).astype('float32') / 255.0


def filtrar_dataset(X_imgs, y, rotulos_alvo, mapa_nomes=None):
    """Filtra as amostras cujo rótulo esteja em `rotulos_alvo`, achata e normaliza.
    `mapa_nomes`, se informado, converte o rótulo numérico original em um nome legível
    (ex.: 1 -> 'A')."""
    mascara = np.isin(y, rotulos_alvo)
    X_filtrado = achatar_e_normalizar(X_imgs[mascara])
    y_filtrado = y[mascara]
    if mapa_nomes is not None:
        y_filtrado = np.array([mapa_nomes[v] for v in y_filtrado])
    return X_filtrado, y_filtrado


def subamostrar_estratificado(X, y, n_amostras, seed=42):
    """Reduz o dataset mantendo a proporção de classes, para acelerar o treino."""
    if n_amostras is None or n_amostras >= len(X):
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X, y, train_size=n_amostras, stratify=y, random_state=seed
    )
    return X_sub, y_sub


# Tamanhos máximos de amostra (ajuste para cima se tiver tempo/GPU disponível)
N_TREINO_MAX = 6000
N_TESTE_MAX = 1500


In [ ]:
# --- 3.1 Binário Verdadeiro / Falso (letras V e F) ---
rotulo_F = [r for r, letra in mapa_letras.items() if letra == 'F'][0]
rotulo_V = [r for r, letra in mapa_letras.items() if letra == 'V'][0]
mapa_vf = {rotulo_F: 'Falso', rotulo_V: 'Verdadeiro'}

X_train_vf, y_train_vf = filtrar_dataset(letters_train_x, letters_train_y, [rotulo_F, rotulo_V], mapa_vf)
X_test_vf, y_test_vf = filtrar_dataset(letters_test_x, letters_test_y, [rotulo_F, rotulo_V], mapa_vf)
X_train_vf, y_train_vf = subamostrar_estratificado(X_train_vf, y_train_vf, N_TREINO_MAX)
X_test_vf, y_test_vf = subamostrar_estratificado(X_test_vf, y_test_vf, N_TESTE_MAX)

# --- 3.2 Multiclasse dígitos 1 a 5 ---
rotulos_1a5 = [1, 2, 3, 4, 5]
X_train_15, y_train_15 = filtrar_dataset(digits_train_x, digits_train_y, rotulos_1a5)
X_test_15, y_test_15 = filtrar_dataset(digits_test_x, digits_test_y, rotulos_1a5)
X_train_15, y_train_15 = subamostrar_estratificado(X_train_15, y_train_15, N_TREINO_MAX)
X_test_15, y_test_15 = subamostrar_estratificado(X_test_15, y_test_15, N_TESTE_MAX)

# --- 3.3 Multiclasse letras A a E ---
rotulos_ae = [r for r, letra in mapa_letras.items() if letra in ['A', 'B', 'C', 'D', 'E']]
mapa_ae = {r: mapa_letras[r] for r in rotulos_ae}
X_train_ae, y_train_ae = filtrar_dataset(letters_train_x, letters_train_y, rotulos_ae, mapa_ae)
X_test_ae, y_test_ae = filtrar_dataset(letters_test_x, letters_test_y, rotulos_ae, mapa_ae)
X_train_ae, y_train_ae = subamostrar_estratificado(X_train_ae, y_train_ae, N_TREINO_MAX)
X_test_ae, y_test_ae = subamostrar_estratificado(X_test_ae, y_test_ae, N_TESTE_MAX)

datasets = {
    'binario_VF': (X_train_vf, X_test_vf, y_train_vf, y_test_vf),
    'multiclasse_1a5': (X_train_15, X_test_15, y_train_15, y_test_15),
    'multiclasse_AaE': (X_train_ae, X_test_ae, y_train_ae, y_test_ae),
}

for nome, (Xtr, Xte, ytr, yte) in datasets.items():
    print(f'{nome}: treino={Xtr.shape}, teste={Xte.shape}, classes={sorted(set(ytr))}')


## 4. Definir os 5 classificadores

Cada modelo é um `Pipeline` (com `StandardScaler` quando faz sentido) para evitar
vazamento de dados entre treino e teste.

In [ ]:
def construir_modelos():
    return {
        'RandomForest': Pipeline([
            ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ]),
        'NaiveBayes': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', GaussianNB()),
        ]),
        'RegressaoLogistica': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1000, n_jobs=-1)),
        ]),
        'SVM': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(kernel='rbf', probability=True, random_state=42)),
        ]),
        'MLP': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)),
        ]),
    }


## 5. Treinar, prever e gerar as métricas (`classification_report` + CSV)

Para cada combinação (dataset x modelo):
- treina o pipeline;
- prevê no treino (para medir overfitting) e no teste;
- salva um `.csv` do `classification_report` do teste;
- guarda o modelo treinado em memória para uso posterior.

In [ ]:
os.makedirs('relatorios', exist_ok=True)

resultados = []
modelos_treinados = {}

for nome_dataset, (X_tr, X_te, y_tr, y_te) in datasets.items():
    modelos_treinados[nome_dataset] = {}
    for nome_modelo, pipeline in construir_modelos().items():
        print(f'Treinando {nome_modelo} em {nome_dataset}...')
        pipeline.fit(X_tr, y_tr)

        y_pred_train = pipeline.predict(X_tr)
        y_pred_test = pipeline.predict(X_te)

        acc_train = accuracy_score(y_tr, y_pred_train)
        acc_test = accuracy_score(y_te, y_pred_test)

        relatorio_teste = classification_report(y_te, y_pred_test, output_dict=True, zero_division=0)
        df_relatorio = pd.DataFrame(relatorio_teste).transpose()
        caminho_csv = f'relatorios/{nome_dataset}_{nome_modelo}_classification_report.csv'
        df_relatorio.to_csv(caminho_csv, encoding='utf-8')

        resultados.append({
            'dataset': nome_dataset,
            'modelo': nome_modelo,
            'acuracia_treino': acc_train,
            'acuracia_teste': acc_test,
            'gap_treino_teste': acc_train - acc_test,
            'f1_macro_teste': relatorio_teste['macro avg']['f1-score'],
        })

        modelos_treinados[nome_dataset][nome_modelo] = pipeline

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv('resumo_resultados.csv', index=False)
df_resultados


## 6. Ranking dos modelos

Critério: priorizar **alta acurácia no teste** e **pequena diferença entre
acurácia de treino e teste** (`gap_treino_teste` pequeno = pouco overfitting/underfitting).

`score_ranking = acuracia_teste - |gap_treino_teste|`

In [ ]:
df_resultados['score_ranking'] = df_resultados['acuracia_teste'] - df_resultados['gap_treino_teste'].abs()

ranking = df_resultados.sort_values(['dataset', 'score_ranking'], ascending=[True, False]).reset_index(drop=True)
ranking.to_csv('ranking_modelos.csv', index=False)
ranking


In [ ]:
melhor_por_dataset = df_resultados.loc[df_resultados.groupby('dataset')['score_ranking'].idxmax()]
melhor_por_dataset


### Justificativa (preencher após ver os números reais)

> Para cada dataset, o modelo escolhido foi o de **maior `score_ranking`**, ou seja,
> o que combina boa acurácia de teste com uma diferença pequena entre treino e teste.
> Um modelo com acurácia de treino muito acima da de teste indica **overfitting**
> (decorou o treino, generaliza mal); um modelo com as duas acurácias baixas indica
> **underfitting**. Descreva aqui, com os números da tabela `melhor_por_dataset`,
> qual modelo venceu em cada um dos 3 datasets e por quê.

## 7. Salvar os melhores modelos para o app Streamlit

Os arquivos `.joblib` gerados aqui devem ser copiados para a pasta `app_streamlit/`
(mesma pasta do `app.py`) antes de rodar o data app.

In [ ]:
for _, linha in melhor_por_dataset.iterrows():
    nome_dataset = linha['dataset']
    nome_modelo = linha['modelo']
    pipeline = modelos_treinados[nome_dataset][nome_modelo]
    caminho = f'modelo_{nome_dataset}.joblib'
    joblib.dump(pipeline, caminho)
    print('Salvo:', caminho, '-> modelo:', nome_modelo)


In [ ]:
# Se estiver no Colab, baixe os 3 arquivos .joblib para o seu computador
# (depois mova-os para a pasta app_streamlit/ do projeto local).
try:
    from google.colab import files
    for _, linha in melhor_por_dataset.iterrows():
        files.download(f"modelo_{linha['dataset']}.joblib")
except ImportError:
    print('Não está rodando no Colab: os arquivos .joblib já estão na pasta local do notebook.')


## 8. Próximo passo: app Streamlit

Use o arquivo `app_streamlit/app.py` (fora deste notebook, roda localmente) com os
3 modelos `.joblib` salvos acima. Ele mostra 3 canvas (`streamlit_mnist_canvas`):
V/F, 1 a 5 e A a E, cada um acionando o modelo correspondente.

Para rodar localmente:
```bash
cd app_streamlit
pip install -r requirements.txt
streamlit run app.py
```
